In [21]:
import ollama

## 1. Loading the dataset

In [2]:
dataset = []
with open('cat-facts.txt', 'r') as file:
  dataset = file.readlines()
  print(f'The loaded dataset contains {len(dataset)} entries.')

The loaded dataset contains 150 entries.


In [5]:
for entry in dataset[:3]:
    print(entry.strip())

On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
Unlike dogs, cats do not have a sweet tooth. Scientists believe this is due to a mutation in a key taste receptor.
When a cat chases its prey, it keeps its head level. Dogs and humans bob their heads up and down.


## 2. Define the vector database

In [ ]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'
VECTOR_DB = []

In [7]:
def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

In [8]:
print('Adding chunks to the vector database...')
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)

Adding chunks to the vector database...


In [12]:
for entry in VECTOR_DB[:3]:
    print(entry)

('On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.\n', [-0.03609412, -0.022022257, 0.046288613, -0.07995774, 0.036077365, -0.014886799, 0.0778245, 0.054345153, -0.013937934, -0.0021585738, -0.019277334, -0.0065506776, -0.057554554, 0.013800884, -0.048231225, 0.039891694, 0.08796751, 0.011429542, -0.032701142, -0.030435419, 0.003829148, 0.027955338, -0.025594188, 0.0001115271, 0.049154025, -0.016156875, -0.008820397, -0.0021448918, 0.003940243, -0.013264699, 0.038277607, -0.029793879, -0.03372232, 0.007034487, 0.026024625, -0.03608568, -0.010429396, -0.03612161, 0.015195706, 0.032245588, -0.033295915, -0.0153117, -0.019713184, 0.01304308, -0.030575413, -0.013332067, -0.0018612732, -0.014753321, 0.02748535, 0.016604796, -0.03332862, 0.00077122607, 0.0050340695, -0.0068562, -0.020287542, 0.04144038, 0.0076216455, -0.0044080676, -0.0067278044, 0.02399276, 0.04380597, 0.0066837175, 0.0069213747, -0.015488447, -

## 3. Implement the retrieval function

In [13]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [14]:
def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  similarities.sort(key=lambda x: x[1], reverse=True)
  return similarities[:top_n]

## 4. Generation phrase

In [15]:
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

In [17]:
print('Retrieving knowledge...')
for chunk, similarity in retrieved_knowledge:
  print(f'(similarity: {similarity:.2f}) {chunk}')

Retrieving knowledge...
(similarity: 0.84) A cat can travel at a top speed of approximately 31 mph (49 km) over a short distance.

(similarity: 0.70) A cat’s heart beats nearly twice as fast as a human heart, at 110 to 140 beats a minute.

(similarity: 0.67) A cat’s jaw can’t move sideways, so a cat can’t chew large chunks of food.



In [18]:
instruction_prompt = f'''You are a helpful chatbot!
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

In [19]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

In [20]:
print('Chatbot responding...')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Chatbot responding...
A cat's top speed is approximately 31 miles per hour (49 kilometers per hour).